In [ ]:
# 1. Inspect local model availability; do not read credentials.
from pathlib import Path
import json, torch, time, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
print('GPU:',torch.cuda.get_device_name(), 'free GiB:',round(torch.cuda.mem_get_info()[0]/2**30,2))
print('Local model configs:',[str(p) for p in Path('models').rglob('model_index.json')])
nb=json.loads(Path('Jev_Attention_Downsampling_Hands.ipynb').read_text())
for c in nb['cells']:
 s=''.join(c.get('source',[]))
 if 'from_pretrained' in s and ('StableDiffusion' in s or 'MODEL_ID' in s):
  print(s[:5000])


In [ ]:
# 2. Independent SD1.5 runtime and an explicit dry-run contract
from PIL import Image
import torch.nn.functional as F
from diffusers import StableDiffusionPipeline, DDIMScheduler
from diffusers.models.attention_processor import AttnProcessor2_0
D_OUT=Path('results')/time.strftime('four_methods_dry_%Y%m%d_%H%M%S');D_OUT.mkdir(parents=True)
D_PROMPT='Studio photograph of a single human hand, open palm facing camera, all five fingers spread apart and fully visible, wrist entering from bottom, centered, plain dark gray background, realistic skin, sharp focus'
D_NEGATIVE='extra hands, extra fingers, missing fingers, fused fingers, deformed, text, watermark, blurry, cropped'
D_PIPE=StableDiffusionPipeline.from_pretrained('stable-diffusion-v1-5/stable-diffusion-v1-5',torch_dtype=torch.float16,use_safetensors=True,local_files_only=True).to('cuda')
D_PIPE.set_progress_bar_config(disable=True)
D_PIPE.unet.requires_grad_(False);D_PIPE.vae.requires_grad_(False);D_PIPE.text_encoder.requires_grad_(False)
D_SCHED=DDIMScheduler.from_config(D_PIPE.scheduler.config);D_SCHED.set_timesteps(100,device='cuda')
with torch.no_grad():
 D_TOK=D_PIPE.tokenizer([D_NEGATIVE,D_PROMPT],padding='max_length',max_length=77,truncation=True,return_tensors='pt').to('cuda')
 D_EMB=D_PIPE.text_encoder(D_TOK.input_ids)[0].detach()
D_ORIGINAL=dict(D_PIPE.unet.attn_processors)
print('Original source candidates:',[str(p) for p in Path('results').glob('hands_attention_*/123_jev.png')])
print('Tokens:',list(enumerate(D_PIPE.tokenizer.convert_ids_to_tokens(D_TOK.input_ids[1].tolist()))))
print('Result folder:',D_OUT)
display(Markdown('**Dry run:** same original seed-123 source, matched latent/noise and prompt. PAG; head-wise SoftPAG (SD1.5 adaptation, not the complete HeadHunter search); attention-property self-guidance (explicit diagnostic target, not anatomy supervision); stock Diffusers FreeU. Save every candidate and full computations. Numerical effect is not a quality score.'))


In [ ]:
# 3. Reuse the original seed-123 Jev image and fixed re-noising (no new source selection).
import hashlib, inspect
D_SOURCE_PATHS=sorted(Path('results').glob('hands_attention_*/123_jev.png'))
assert len(D_SOURCE_PATHS)==1, D_SOURCE_PATHS
D_SOURCE=Image.open(D_SOURCE_PATHS[0]).convert('RGB')
@torch.no_grad()
def d_decode(z):
 x=(D_PIPE.vae.decode(z/D_PIPE.vae.config.scaling_factor).sample/2+.5).clamp(0,1)
 return Image.fromarray((x[0].float().cpu().permute(1,2,0).numpy()*255).round().astype('uint8'))
with torch.no_grad():
 pixels=torch.from_numpy(np.asarray(D_SOURCE).copy()).permute(2,0,1)[None].to('cuda',torch.float16)/127.5-1
 D_CLEAN=D_PIPE.vae.encode(pixels).latent_dist.mode()*D_PIPE.vae.config.scaling_factor
 D_NOISE=torch.randn(D_CLEAN.shape,generator=torch.Generator(device='cuda').manual_seed(123100100),device='cuda',dtype=torch.float16)
 D_Z0=D_SCHED.add_noise(D_CLEAN,D_NOISE,D_SCHED.timesteps[50:51])
D_SELF='mid_block.attentions.0.transformer_blocks.0.attn1.processor'
D_CROSS='down_blocks.2.attentions.1.transformer_blocks.0.attn2.processor'
D_WORDS=D_PIPE.tokenizer.convert_ids_to_tokens(D_TOK.input_ids[1].tolist())
D_HAND=[i for i,w in enumerate(D_WORDS) if w=='hand</w>'];assert len(D_HAND)==1
D_MANIFEST={'source':str(D_SOURCE_PATHS[0]),'source_sha256':hashlib.sha256(D_SOURCE.tobytes()).hexdigest(),'seed':123,'renoise_seed':123100100,'schedule_steps':100,'start':50,'checkpoints':[50,65,80],'cfg':7.5,'prompt':D_PROMPT,'negative':D_NEGATIVE,'PAG':{'module':D_SELF,'scale':1.0},'SoftPAG':{'module':D_SELF,'heads':list(range(8)),'interpolation':.5,'scale':1.0,'status':'SD1.5 head perturbation diagnostic; not full HeadHunter optimization'},'self_guidance':{'module':D_CROSS,'token':D_HAND,'target':'attention-derived hand size +10%; centroid preserved','status':'SD1.5 single-layer adaptation; not a finger-correctness objective'},'FreeU':{'b1':1.2,'b2':1.4,'s1':.9,'s2':.2},'interpretation':'Effect sizes and gradient norms are not image-quality scores'}
(D_OUT/'manifest.json').write_text(json.dumps(D_MANIFEST,indent=2));D_SOURCE.save(D_OUT/'original_seed123.png')
print('Source:',D_SOURCE_PATHS[0]);print('Self-attention probe:',D_SELF);print('Cross-attention probe:',D_CROSS,'hand token:',D_HAND)
display(D_SOURCE.resize((384,384)))
print('Installed FreeU implementation:')
from diffusers.utils.torch_utils import apply_freeu
print(inspect.getsource(apply_freeu))


In [ ]:
# 4. Instrument actual attention computations and FreeU feature transformations.
from unittest.mock import patch
D_MODE={};D_TRACE={};D_CALLS=0
class DStopCapture(Exception): pass
def d_rms(x): return float(x.detach().float().square().mean().sqrt())
class DAttention:
 def __init__(self,name): self.name=name
 def __call__(self,attn,hidden_states,encoder_hidden_states=None,attention_mask=None,temb=None,*args,**kwargs):
  residual=hidden_states;ndim=hidden_states.ndim
  if attn.spatial_norm is not None: hidden_states=attn.spatial_norm(hidden_states,temb)
  if ndim==4:
   b,ch,hh,ww=hidden_states.shape;hidden_states=hidden_states.view(b,ch,hh*ww).transpose(1,2)
  b,n,ch=hidden_states.shape
  if attn.group_norm is not None: hidden_states=attn.group_norm(hidden_states.transpose(1,2)).transpose(1,2)
  enc=hidden_states if encoder_hidden_states is None else encoder_hidden_states
  if encoder_hidden_states is not None and attn.norm_cross: enc=attn.norm_encoder_hidden_states(enc)
  h=attn.heads;d=attn.inner_dim//h
  q=attn.to_q(hidden_states).view(b,n,h,d).transpose(1,2)
  k=attn.to_k(enc).view(b,-1,h,d).transpose(1,2);v=attn.to_v(enc).view(b,-1,h,d).transpose(1,2)
  assert attention_mask is None, 'This diagnostic assumes the observed unmasked SD1.5 calls.'
  if self.name==D_CROSS:
   probs=(q[-1].float()@k[-1].float().transpose(-1,-2)*attn.scale).softmax(-1)
   D_TRACE['cross_maps']=probs.mean(0).transpose(0,1).reshape(77,16,16)
   if D_MODE.get('sg_only'): raise DStopCapture()
  out=F.scaled_dot_product_attention(q,k,v,dropout_p=0.)
  if self.name==D_SELF:
   probs=(q[-1].float()@k[-1].float().transpose(-1,-2)*attn.scale).softmax(-1)
   D_TRACE['self']={'grid':int(n**.5),'heads':h,'q':q[-1].detach().cpu(),'k':k[-1].detach().cpu(),'v':v[-1].detach().cpu(),'attention':probs.detach().cpu(),'head_output':out[-1].detach().cpu()}
   heads=D_MODE.get('heads',[]);u=D_MODE.get('u',0.)
   if heads:
    out=out.clone();out[-1,heads]=(1-u)*out[-1,heads]+u*v[-1,heads]
  out=out.transpose(1,2).reshape(b,n,h*d).to(q.dtype);out=attn.to_out[1](attn.to_out[0](out))
  if ndim==4: out=out.transpose(-1,-2).reshape(b,ch,hh,ww)
  if attn.residual_connection: out=out+residual
  return out/attn.rescale_output_factor
def d_install():
 D_PIPE.unet.set_attn_processor({n:DAttention(n) if n in [D_SELF,D_CROSS] else p for n,p in D_ORIGINAL.items()})
def d_freeu_trace(resolution_idx,hidden_states,res_hidden_states,**kw):
 before_h=hidden_states.detach().clone();before_s=res_hidden_states.detach().clone()
 h,s=apply_freeu(resolution_idx,hidden_states,res_hidden_states,**kw)
 D_TRACE.setdefault('freeu',[]).append({'stage':resolution_idx,'backbone_shape':list(h.shape),'skip_shape':list(s.shape),'backbone_rms_before':d_rms(before_h),'backbone_rms_after':d_rms(h),'skip_rms_before':d_rms(before_s),'skip_rms_after':d_rms(s),'backbone_delta_rms':d_rms(h-before_h),'skip_delta_rms':d_rms(s-before_s)})
 return h,s
@torch.no_grad()
def d_predict(z,i,heads=(),u=0.,freeu=False):
 global D_MODE,D_TRACE,D_CALLS
 D_MODE={'heads':list(heads),'u':u};D_TRACE={};D_CALLS+=1;d_install()
 try:
  if freeu: D_PIPE.unet.enable_freeu(**D_MANIFEST['FreeU'])
  with patch('diffusers.models.unets.unet_2d_blocks.apply_freeu',d_freeu_trace):
   eps=D_PIPE.unet(z.repeat(2,1,1,1),D_SCHED.timesteps[i],encoder_hidden_states=D_EMB).sample
  return eps.chunk(2),D_TRACE
 finally:
  D_PIPE.unet.disable_freeu();D_PIPE.unet.set_attn_processor(dict(D_ORIGINAL))
def d_cfg(pair): return pair[0]+7.5*(pair[1]-pair[0])
def d_plain(z,i): return d_cfg(d_predict(z,i)[0])
with torch.no_grad():
 raw=D_PIPE.unet(D_Z0.repeat(2,1,1,1),D_SCHED.timesteps[50],encoder_hidden_states=D_EMB).sample
 neutral,trace=d_predict(D_Z0,50)
 err=float((torch.cat(neutral)-raw).abs().max());assert err<.02,err
 pert,_=d_predict(D_Z0,50,range(8),1.)
 assert d_rms(pert[0]-neutral[0])==0, 'PAG must leave the unconditional branch unchanged'
 assert d_rms(pert[1]-neutral[1])>0
print('Neutral wrapper max error:',err,'PAG conditional delta RMS:',d_rms(neutral[1]-pert[1]))
print('Measured self grid:',trace['self']['grid'],'cross map shape:',trace['cross_maps'].shape)


In [ ]:
# 5. Self-guidance: differentiable internal property, real latent gradient, explicit target.
# Adapted from Self-Guidance equations 4, 5, 16 (single SD1.5 layer).
# Target: hand-token attention area +10%, original centroid. Not five-finger supervision.
def d_normalize(x): return (x-x.amin())/(x.amax()-x.amin()).clamp_min(1e-6)
def d_properties(m):
 mask=d_normalize(torch.sigmoid(10*(d_normalize(m)-.5)))
 yy,xx=torch.meshgrid(torch.linspace(0,1,m.shape[0],device=m.device),torch.linspace(0,1,m.shape[1],device=m.device),indexing='ij')
 mass=m.sum().clamp_min(1e-6)
 return torch.stack([mask.mean(),(m*xx).sum()/mass,(m*yy).sum()/mass]),mask
def d_sg_map(z,i):
 global D_MODE,D_TRACE,D_CALLS
 D_MODE={'sg_only':True};D_TRACE={};D_CALLS+=1;d_install()
 try:
  D_PIPE.unet(z,D_SCHED.timesteps[i],encoder_hidden_states=D_EMB[1:])
 except DStopCapture: pass
 finally: D_PIPE.unet.set_attn_processor(dict(D_ORIGINAL))
 return D_TRACE['cross_maps'][D_HAND].mean(0)
def d_energy(props,target):
 return ((props[0]-target[0])/target[0].clamp_min(.01)).square()+10*(props[1:]-target[1:]).square().sum()
def d_sg(z,i,target):
 with torch.enable_grad():
  zz=z.detach().clone().requires_grad_(True);m=d_sg_map(zz,i);props,mask=d_properties(m)
  energy=d_energy(props,target);grad=torch.autograd.grad(energy*1024,zz)[0].float()/1024
 assert torch.isfinite(grad).all() and d_rms(grad)>0
 return grad.detach(),{'energy':float(energy.detach()),'properties':props.detach().cpu().tolist(),'target':target.cpu().tolist(),'gradient_rms':d_rms(grad),'map':m.detach().cpu(),'mask':mask.detach().cpu()}
with torch.no_grad():
 p0,m0=d_properties(d_sg_map(D_Z0,50));target=p0.detach().clone();target[0]*=1.1
grad,sgtest=d_sg(D_Z0,50,target)
direction=grad/grad.square().mean().sqrt().clamp_min(1e-12)
with torch.no_grad():
 minus,_=d_properties(d_sg_map((D_Z0.float()-.01*direction).half(),50))
 plus,_=d_properties(d_sg_map((D_Z0.float()+.01*direction).half(),50))
 em=float(d_energy(minus,target));ep=float(d_energy(plus,target))
assert em<ep,('Gradient sign check failed',em,ep)
print('Original [area,cx,cy]:',p0.cpu().tolist(),'target:',target.cpu().tolist())
print('Energy -gradient / original / +gradient:',em,sgtest['energy'],ep,'gradient RMS:',d_rms(grad))
D_SG_PREFLIGHT={'energy_minus':em,'energy_original':sgtest['energy'],'energy_plus':ep,'grad_rms':d_rms(grad)}
(D_OUT/'preflight.json').write_text(json.dumps({'neutral_error':err,'self_guidance':D_SG_PREFLIGHT},indent=2))
print('No bounding boxes or anatomical labels were assigned to heads. All numerical calculations above are PyTorch operations.')


In [ ]:
# 6. Fixed reference trajectory: matched checkpoints, no image selection.
D_CHECKPOINTS={};D_BASE_FRAMES={};D_BASE_PROPS={}
z=D_Z0.clone();started=time.time()
for i in range(50,100):
 with torch.no_grad():
  if i in [50,65,80]: D_CHECKPOINTS[i]=z.detach().clone()
  pair,tr=d_predict(z,i);eps=d_cfg(pair)
  D_BASE_PROPS[i]=d_properties(tr['cross_maps'][D_HAND].mean(0))[0].detach().clone()
  st=D_SCHED.step(eps,D_SCHED.timesteps[i],z,eta=0.);z=st.prev_sample
  if i in [50,59,64,69,79,89,99]:
   im=d_decode(st.pred_original_sample);im.save(D_OUT/f'reference_{i+1:03}.png');D_BASE_FRAMES[i+1]=im
 if i%10==9: print('Reference step',i+1,'elapsed',round(time.time()-started,1),'s',flush=True)
D_BASE_FINAL=d_decode(z);D_BASE_FINAL.save(D_OUT/'reference_final.png')
torch.save({'start':D_Z0.cpu(),'checkpoints':{k:v.cpu() for k,v in D_CHECKPOINTS.items()},'source_latent':D_CLEAN.cpu()},D_OUT/'matched_states.pt')
fig,axes=plt.subplots(1,len(D_BASE_FRAMES),figsize=(18,3))
for ax,(i,im) in zip(axes,D_BASE_FRAMES.items()): ax.imshow(im);ax.set_title(f'step {i}');ax.axis('off')
plt.tight_layout();plt.show()
print('Saved checkpoints:',list(D_CHECKPOINTS),'UNet calls:',D_CALLS)


In [ ]:
# 7. Compute all candidate directions; retain the complete head scan, not just the largest effect.
def d_cpu(x):
 if torch.is_tensor(x): return x.detach().cpu()
 if isinstance(x,dict): return {k:d_cpu(v) for k,v in x.items()}
 if isinstance(x,(list,tuple)): return [d_cpu(v) for v in x]
 return x
def d_cos(a,b): return float(F.cosine_similarity(a.float().flatten()[None],b.float().flatten()[None]).item())
def d_probe(i):
 z=D_CHECKPOINTS[i];pair,tr=d_predict(z,i);base=d_cfg(pair);candidates={};details={};rows=[]
 pp,pt=d_predict(z,i,range(8),1.);candidates['PAG']=pair[1]-pp[1]
 details['PAG']={'normal_conditional':pair[1],'perturbed_conditional':pp[1],'formula':'eps_cfg + (eps_cond - eps_pert_cond)'}
 for h in range(8):
  hp,ht=d_predict(z,i,[h],.5);candidates[f'SoftPAG_h{h}']=pair[1]-hp[1]
  details[f'SoftPAG_h{h}']={'head':h,'u':.5,'perturbed_conditional':hp[1]}
 jp,jt=d_predict(z,i,[0,1],.5);candidates['SoftPAG_h0_h1']=pair[1]-jp[1]
 details['SoftPAG_h0_h1']={'joint_vs_sum_rms':d_rms(candidates['SoftPAG_h0_h1']-candidates['SoftPAG_h0']-candidates['SoftPAG_h1'])}
 fp,ft=d_predict(z,i,freeu=True);candidates['FreeU']=d_cfg(fp)-base;details['FreeU']=ft['freeu']
 for factor,label in [(1.1,'SG_area_plus10'),(.9,'SG_area_minus10')]:
  target=D_BASE_PROPS[i].clone();target[0]*=factor;g,info=d_sg(z,i,target)
  sigma=(1-D_SCHED.alphas_cumprod[D_SCHED.timesteps[i]]).sqrt().to(g)
  raw=sigma*g;gain=.05*d_rms(base)/max(d_rms(raw),1e-12)
  candidates[label]=raw*gain;details[label]={**info,'raw_sigma_gradient':raw,'diagnostic_gain':gain,'delta_rms_fraction_of_eps':.05}
 views={};after_states={};before=d_decode(D_SCHED.step(base,D_SCHED.timesteps[i],z,eta=0.).pred_original_sample)
 for label,delta in {'reference':torch.zeros_like(base),**candidates}.items():
  with torch.no_grad():
   eps=(base.float()+delta.float()).to(z.dtype);step=D_SCHED.step(eps,D_SCHED.timesteps[i],z,eta=0.)
   zz=step.prev_sample
   for j in range(i+1,i+4):
    step=D_SCHED.step(d_plain(zz,j),D_SCHED.timesteps[j],zz,eta=0.);zz=step.prev_sample
   im=d_decode(step.pred_original_sample);views[label]=im;after_states[label]=zz.cpu()
   im.save(D_OUT/f'probe_{i}_{label}_after4.png')
  if label!='reference':
   arr=np.asarray(im,dtype=np.float32)/255;ref=np.asarray(views['reference'],dtype=np.float32)/255
   rows.append({'checkpoint':i,'method':label,'delta_eps_rms':d_rms(delta),'relative_eps_rms':d_rms(delta)/d_rms(base),'cosine_with_PAG':d_cos(delta,candidates['PAG']),'rgb_change_after4':float(np.abs(arr-ref).mean()),'unconditional_change':'FreeU changes both branches' if label=='FreeU' else 'unchanged'})
 result={'i':i,'base_eps':base,'normal_pair':pair,'trace':tr,'directions':candidates,'details':details,'after4_states':after_states,'rows':rows}
 torch.save(d_cpu(result),D_OUT/f'computations_{i}.pt')
 df=pd.DataFrame(rows);df.to_csv(D_OUT/f'diagnostics_{i}.csv',index=False);display(df.round(6))
 fig,axes=plt.subplots(4,4,figsize=(14,14))
 for ax in axes.flat: ax.axis('off')
 for ax,(label,im) in zip(axes.flat,views.items()): ax.imshow(im);ax.set_title(label)
 fig.suptitle(f'Checkpoint {i}: ONE intervention, then 3 ordinary steps; all candidates shown',fontsize=13)
 plt.tight_layout();fig.savefig(D_OUT/f'all_candidates_{i}.png',dpi=110);plt.show()
 return result,views
D_PROBES={};D_PROBE_VIEWS={}
for i in [50,65,80]:
 print('Probing checkpoint',i,flush=True);D_PROBES[i],D_PROBE_VIEWS[i]=d_probe(i)
 print('Checkpoint complete',i,'UNet calls',D_CALLS,flush=True)


In [ ]:
# 8. Numerical repeat floor and maps: magnitudes measure effects, not quality.
D_REPEAT=[]
for i in [50,65,80]:
 with torch.no_grad():
  zz=D_CHECKPOINTS[i].clone();eps0=d_plain(zz,i)
  eps_repeat=d_plain(zz,i)
  for j in range(i,i+4):
   eps=eps0 if j==i else d_plain(zz,j)
   st=D_SCHED.step(eps,D_SCHED.timesteps[j],zz,eta=0.);zz=st.prev_sample
  im=d_decode(st.pred_original_sample)
  rgb=float(np.abs(np.asarray(im,dtype=float)-np.asarray(D_PROBE_VIEWS[i]['reference'],dtype=float)).mean()/255)
  D_REPEAT.append({'checkpoint':i,'same_state_eps_repeat_rms':d_rms(eps_repeat-eps0),'stored_vs_repeat_eps_rms':d_rms(D_PROBES[i]['base_eps']-eps0),'reference_rgb_repeat_after4':rgb})
print('Repeat floor (candidate RGB effects should be read alongside this):');display(pd.DataFrame(D_REPEAT))
(D_OUT/'repeat_floor.json').write_text(json.dumps(D_REPEAT,indent=2))
fig,axes=plt.subplots(3,5,figsize=(17,10))
D_COSINES={}
for row,i in enumerate([50,65,80]):
 r=D_PROBES[i];hand=r['trace']['cross_maps'][D_HAND].mean(0).detach().cpu();axes[row,0].imshow(hand,cmap='magma');axes[row,0].set_title(f'{i}: hand-token attention')
 for col,name in enumerate(['PAG','FreeU','SG_area_plus10'],1):
  dm=r['directions'][name][0].float().square().mean(0).sqrt().cpu();axes[row,col].imshow(dm,cmap='inferno');axes[row,col].set_title(f'{name}: spatial delta RMS')
 hs=torch.stack([r['directions'][f'SoftPAG_h{h}'].float().flatten().cpu() for h in range(8)]);hn=F.normalize(hs,dim=1);cos=hn@hn.T;D_COSINES[i]=cos
 axes[row,4].imshow(cos,cmap='coolwarm',vmin=-1,vmax=1);axes[row,4].set_title('Head direction cosine');axes[row,4].set_xticks(range(8));axes[row,4].set_yticks(range(8))
 for ax in axes[row,:4]:ax.axis('off')
plt.tight_layout();fig.savefig(D_OUT/'internal_computation_maps.png',dpi=130);plt.show()
print('FreeU backbone/skip transformations at checkpoint 50:');display(pd.DataFrame(D_PROBES[50]['details']['FreeU']).round(5))
print('Joint head nonadditivity:',{i:D_PROBES[i]['details']['SoftPAG_h0_h1'] for i in D_PROBES})
D_SUMMARY=pd.concat([pd.DataFrame(D_PROBES[i]['rows']) for i in D_PROBES],ignore_index=True)
D_SUMMARY.to_csv(D_OUT/'all_computations.csv',index=False)


In [ ]:
# 9. Four complete continuations, with the first EIGHT steps intervened.
# Settings are declared before viewing endpoints. No best-of-head or image selection.
D_ROLLOUT_CONFIG={'start':50,'intervene_until_exclusive':58,'PAG_scale':1.,'SoftPAG_heads':[0,1],'SoftPAG_u':.5,'SG_area_factor':1.1,'SG_delta_fraction':.05,'FreeU':D_MANIFEST['FreeU']}
(D_OUT/'rollout_config.json').write_text(json.dumps(D_ROLLOUT_CONFIG,indent=2))
D_ENDPOINTS={};D_PROGRESS={};D_ROLLOUT_LOG=[]
for method in ['PAG','SoftPAG_h0_h1','SG_area_plus10','FreeU']:
 zz=D_Z0.clone();frames={};begun=time.time();print('Starting',method,flush=True)
 for i in range(50,100):
  with torch.no_grad():
   pair,tr=d_predict(zz,i);eps=d_cfg(pair);base=eps.clone()
   if i<58 and method=='PAG':
    pp,_=d_predict(zz,i,range(8),1.);eps=eps+(pair[1]-pp[1])
   elif i<58 and method=='SoftPAG_h0_h1':
    pp,_=d_predict(zz,i,[0,1],.5);eps=eps+(pair[1]-pp[1])
   elif i<58 and method=='FreeU': eps=d_cfg(d_predict(zz,i,freeu=True)[0])
  if i<58 and method=='SG_area_plus10':
   target=D_BASE_PROPS[i].clone();target[0]*=1.1;g,info=d_sg(zz,i,target)
   # Bounded normalized energy guidance is our declared adaptation, not the paper's Imagen weight.
   delta=g*(.05*d_rms(base)/max(d_rms(g),1e-12));eps=(base.float()+delta).half()
  with torch.no_grad():
   D_ROLLOUT_LOG.append({'method':method,'step':i,'intervened':i<58,'delta_eps_rms':d_rms(eps-base)})
   st=D_SCHED.step(eps,D_SCHED.timesteps[i],zz,eta=0.);zz=st.prev_sample
   if i in [50,53,57,64,79,99]:
    im=d_decode(st.pred_original_sample);im.save(D_OUT/f'rollout_{method}_{i+1:03}.png');frames[i+1]=im
  if i%10==9: print(method,'step',i+1,round(time.time()-begun,1),'s',flush=True)
 D_ENDPOINTS[method]=d_decode(zz);D_PROGRESS[method]=frames
 D_ENDPOINTS[method].save(D_OUT/f'final_{method}.png')
 fig,axes=plt.subplots(1,len(frames),figsize=(17,3))
 for ax,(step,im) in zip(axes,frames.items()): ax.imshow(im);ax.set_title(f'{method}: {step}');ax.axis('off')
 plt.tight_layout();fig.savefig(D_OUT/f'progression_{method}.png',dpi=120);plt.show()
pd.DataFrame(D_ROLLOUT_LOG).to_csv(D_OUT/'rollout_trace.csv',index=False)
fig,axes=plt.subplots(1,6,figsize=(19,4))
for ax,(name,im) in zip(axes,{'source':D_SOURCE,'ordinary continuation':D_BASE_FINAL,**D_ENDPOINTS}.items()): ax.imshow(im);ax.set_title(name);ax.axis('off')
plt.tight_layout();fig.savefig(D_OUT/'all_endpoints.png',dpi=160);plt.show()
print('All four completed; every progression retained. This is a one-image mechanism test, not a quality benchmark.')


In [ ]:
# 10. Verbatim Jev prompts: interpret measured computations and choose the next probe.
# These calls do not retroactively choose the displayed images or claim a winner.
D_JEV_INSTRUCTIONS='''You are scheduling measurements for an SD1.5 hand-generation experiment. Each record comes from an actual intervention at exactly the same latent, timestep, prompt and noise. Your job is to choose the next experiment that would best distinguish plausible explanations of the measured response.
Read the numerical records directly. The head cosine matrix compares changes in the final noise prediction. Negative cosine means opposing directions; neither sign indicates better anatomy. RMS measures change magnitude, not quality. Attention maps show association with a text token, not a segmentation ground truth. FreeU rows describe the features actually transformed. Self-guidance has a declared attention-area target; lowering its loss only establishes progress on that target.
You have no direct visual input in this call. The image-quality outcome is unassessed. Do not infer a finger count, an anatomical role for a head, or a preferred final image. Use differences between mechanisms, spatial footprints, interaction residuals and repeat controls to choose a discriminating follow-up. A null result is informative. Each question is evaluated independently; do not assume another answer has already been chosen.'''
D_JEV_QUESTIONS={
 'PAG_next':{'type':'choice','instructions':D_JEV_INSTRUCTIONS+' Choose the next PAG measurement.', 'criteria':{'strength_sweep':'Hold layer and latent fixed; compare guidance gains 0, 1, 3.','layer_sweep':'Hold gain and latent fixed; perturb analogous self-attention modules at other scales.','repeat':'Repeat the identical normal and perturbed predictions to test reproducibility.'}},
 'SoftPAG_next':{'type':'choice','instructions':D_JEV_INSTRUCTIONS+' Choose the next head-level measurement.', 'criteria':{'opposing_heads':'Separately continue heads with the most opposed measured output directions.','composition':'Compare the measured joint-head intervention with the sum of its individual output directions.','other_layers':'Measure head responses at additional spatial scales.'}},
 'SelfGuidance_next':{'type':'choice','instructions':D_JEV_INSTRUCTIONS+' Choose the next self-guidance measurement.', 'criteria':{'target_response':'Track declared area and centroid errors after both +10% and -10% area guidance.','representation':'Inspect hand, fingers and palm text-token maps across layers before assigning structural meaning.','centroid':'Test an explicitly declared centroid displacement while preserving attention area.'}},
 'FreeU_next':{'type':'choice','instructions':D_JEV_INSTRUCTIONS+' Choose the next FreeU measurement.', 'criteria':{'factorial':'Separate backbone-only, skip-only, joint and neutral settings at the same state.','stage_sweep':'Apply the same feature scaling separately at each eligible upsampling stage.','strength_sweep':'Vary the strength of the current joint configuration.'}},
 'quality_claim':{'type':'choice','instructions':D_JEV_INSTRUCTIONS+' What conclusion about final hand quality is supported?', 'criteria':{'improved':'The supplied evidence establishes better hand anatomy.','worse':'The supplied evidence establishes worse hand anatomy.','unassessed':'Numerical responses establish effects, but do not establish anatomical improvement or degradation.'}}}
print(D_JEV_INSTRUCTIONS)
for k,q in D_JEV_QUESTIONS.items(): print('\n'+k+': '+q['instructions'].split('Each question is evaluated independently; do not assume another answer has already been chosen.')[-1],q['criteria'])
(D_OUT/'jev_verbatim_prompts.json').write_text(json.dumps(D_JEV_QUESTIONS,indent=2))


In [ ]:
# 11. Three concurrent Jev dry-run reviews; save exact requests and responses.
import requests
from dotenv import dotenv_values
from concurrent.futures import ThreadPoolExecutor
_d_env=dotenv_values('man.env')
_d_keys=[v for k,v in _d_env.items() if ('typesafe' in k.lower().replace('_','') or 'jev' in k.lower()) and v]
assert len(_d_keys)==1, 'Expected one Typesafe/Jev credential in man.env; values are never printed.'
_d_key=_d_keys[0]
def d_jev_review(i):
 r=D_PROBES[i]
 spatial={}
 for name,delta in r['directions'].items():
  dm=delta[0].float().square().mean(0).sqrt()[None,None]
  spatial[name]=F.adaptive_avg_pool2d(dm,(4,4))[0,0].cpu().round(decimals=6).tolist()
 state={'checkpoint':i,'noise_timestep':int(D_SCHED.timesteps[i]),'prompt':D_PROMPT,'measurements':r['rows'],'repeat_control':next(x for x in D_REPEAT if x['checkpoint']==i),'head_cosine_matrix':D_COSINES[i].round(decimals=4).tolist(),'head_order':list(range(8)),'spatial_direction_rms_4x4':spatial,'joint_head_interaction':r['details']['SoftPAG_h0_h1'],'FreeU_features':r['details']['FreeU'],'self_guidance':{k:{a:b for a,b in r['details'][k].items() if not torch.is_tensor(b)} for k in ['SG_area_plus10','SG_area_minus10']},'image_quality':'No validated visual review supplied; all candidate images saved for human inspection.'}
 payload={'model':'jev-1.13.0','state':state,'questions':D_JEV_QUESTIONS}
 blob=json.dumps(payload,indent=2);assert _d_key not in blob
 (D_OUT/f'jev_{i}_request.json').write_text(blob)
 response=requests.post('https://api.typesafe.ai/v1/systemone',headers={'Authorization':'Bearer '+_d_key},json=payload,timeout=90)
 if response.status_code!=200: return {'checkpoint':i,'error':f'HTTP {response.status_code}'}
 result=response.json();(D_OUT/f'jev_{i}_response.json').write_text(json.dumps(result,indent=2))
 return {'checkpoint':i,'answers':result.get('answers',{}),'usage':result.get('usage',{})}
with ThreadPoolExecutor(max_workers=3) as pool: D_JEV_RESULTS=list(pool.map(d_jev_review,[50,65,80]))
D_JEV_ROWS=[]
for r in D_JEV_RESULTS:
 print('Checkpoint',r['checkpoint'],'error',r.get('error'))
 for q,a in r.get('answers',{}).items():
  D_JEV_ROWS.append({'checkpoint':r['checkpoint'],'question':q,'choice':a.get('choice'),'probabilities':a.get('probabilities')})
display(pd.DataFrame(D_JEV_ROWS));(D_OUT/'jev_dry_run_summary.json').write_text(json.dumps(D_JEV_RESULTS,indent=2))
print('These are proposed follow-up measurements, not executed image-selection decisions.')



In [ ]:
# Credential lookup diagnostic: names only, never values.
print('Available credential names:',list(_d_env))


In [ ]:
# 12. Save a browsable report with all outcomes, tensors, traces and Jev requests.
from IPython.display import HTML
import html
D_FINAL_METRICS=[]
for method,im in D_ENDPOINTS.items():
 a=np.asarray(im,dtype=np.float32)/255;b=np.asarray(D_BASE_FINAL,dtype=np.float32)/255
 D_FINAL_METRICS.append({'method':method,'mean_abs_rgb_vs_ordinary':float(np.abs(a-b).mean()),'changed_pixel_fraction':float(np.any(np.asarray(im)!=np.asarray(D_BASE_FINAL),axis=-1).mean())})
display(pd.DataFrame(D_FINAL_METRICS));print('Jev choices:');display(pd.DataFrame(D_JEV_ROWS)[['checkpoint','question','choice']])
sections=['<h1>Hands: four mechanisms, actual computations</h1><p>Original seed 123; matched re-noising seed 123100100; SD1.5; fixed prompt. Three checkpoint dry runs, every head candidate retained. Four full continuations with intervention only on steps 50-57. No image winner selected. Jev requests choose future measurements, not final images.</p>']
sections+=['<h2>Complete endpoints</h2><img src="all_endpoints.png">',pd.DataFrame(D_FINAL_METRICS).to_html(index=False),'<p>RGB difference measures change, not image quality.</p>','<h2>Internal computations</h2><img src="internal_computation_maps.png">']
for method in D_ENDPOINTS:
 sections.append(f'<h2>{html.escape(method)}: every recorded stage</h2><img src="progression_{method}.png">')
for i in [50,65,80]:
 sections.append(f'<h2>Checkpoint {i}: all 13 interventions and reference</h2><img src="all_candidates_{i}.png"><a href="computations_{i}.pt">Raw computation tensors</a> | <a href="diagnostics_{i}.csv">Measurements CSV</a> | <a href="jev_{i}_request.json">Exact Jev context and prompts</a> | <a href="jev_{i}_response.json">Jev response</a>')
sections+=['<h2>Jev follow-up choices (not executed)</h2>',pd.DataFrame(D_JEV_ROWS)[['checkpoint','question','choice']].to_html(index=False),'<p>The quality question explicitly states that anatomy is unassessed; its answer is not evidence of independent judgment or evidence sensitivity.</p>','<h2>Scope and sources</h2><p>Stock PAG identity perturbation; head-wise SoftPAG adapted to SD1.5, without HeadHunter scoring/search; single-layer attention-property self-guidance with a declared normalized step, not the full Imagen implementation; installed Diffusers FreeU. These are mechanism demonstrations on one existing image.</p><p><a href="https://arxiv.org/abs/2403.17377">PAG</a> | <a href="https://github.com/cvlab-kaist/HeadHunter">HeadHunter / SoftPAG</a> | <a href="https://arxiv.org/html/2306.00986v3">Self-Guidance</a> | <a href="https://github.com/ChenyangSi/FreeU">FreeU</a></p><p><a href="manifest.json">Manifest</a> | <a href="rollout_config.json">Declared continuation settings</a> | <a href="all_computations.csv">All measurements</a> | <a href="repeat_floor.json">Repeat checks</a> | <a href="jev_verbatim_prompts.json">Verbatim prompts</a></p>']
report='<!doctype html><meta charset="utf-8"><title>Hands - four mechanism dry run</title><style>body{font:16px system-ui;max-width:1500px;margin:32px auto;padding:0 24px;background:#f6f7fa;color:#172033}img{max-width:100%;height:auto;background:white}table{border-collapse:collapse;background:white;margin:20px 0}td,th{padding:8px 12px;border:1px solid #ccd3dd}a{color:#185abc}p{line-height:1.6}</style>'+''.join(sections)
(D_OUT/'report.html').write_text(report)
D_REPORT_URL='/files/workspace/crazy_exp/'+str(D_OUT)+'/report.html'
display(HTML(f'<h2>Dry run complete</h2><p><a target="_blank" href="{D_REPORT_URL}">Open full results: images, computations, progressions and Jev choices</a></p><img style="max-width:100%" src="/files/workspace/crazy_exp/{D_OUT}/all_endpoints.png">'))
print('Result folder:',D_OUT,'UNet full/partial forward calls:',D_CALLS)


In [ ]:
# 13. Make the report self-contained (Jupyter's HTML sandbox blocks authenticated subresources).
import base64,re
fig,axes=plt.subplots(2,3,figsize=(12,8))
for ax,(name,im) in zip(axes.flat,{'original source':D_SOURCE,'ordinary continuation':D_BASE_FINAL,**D_ENDPOINTS}.items()):
 ax.imshow(im);ax.set_title(name);ax.axis('off')
plt.tight_layout();fig.savefig(D_OUT/'all_endpoints.png',dpi=130);plt.show()
report=(D_OUT/'report.html').read_text()
def d_embed(match):
 if match.group(1).startswith('data:'): return match.group(0)
 p=D_OUT/match.group(1);encoded=base64.b64encode(p.read_bytes()).decode('ascii')
 return '<img src="data:image/png;base64,'+encoded+'"'
report=re.sub(r'<img src="([^"]+)"',d_embed,report)
(D_OUT/'report.html').write_text(report)
print('Self-contained report saved:',round(len(report)/1e6,2),'MB')
display(HTML(f'<a target="_blank" href="{D_REPORT_URL}">Open verified report with embedded images</a>'))



In [ ]:
# 14. Check whether the self-guidance target survives the four-step probe.
D_SG_OUTCOMES=[]
for i in [50,65,80]:
 for label in ['reference','SG_area_plus10','SG_area_minus10']:
  zz=D_PROBES[i]['after4_states'][label].to('cuda')
  with torch.no_grad(): props,_=d_properties(d_sg_map(zz,i+4))
  factor=1.1 if label=='SG_area_plus10' else (.9 if label=='SG_area_minus10' else 1.)
  target=D_BASE_PROPS[i+4].clone();target[0]*=factor
  D_SG_OUTCOMES.append({'checkpoint':i,'branch':label,'area_after4':float(props[0]),'reference_area_after4':float(D_BASE_PROPS[i+4][0]),'target_area_after4':float(target[0]),'centroid_displacement':float((props[1:]-target[1:]).norm()),'target_energy_after4':float(d_energy(props,target))})
display(pd.DataFrame(D_SG_OUTCOMES).round(6));pd.DataFrame(D_SG_OUTCOMES).to_csv(D_OUT/'self_guidance_target_followup.csv',index=False)
report=(D_OUT/'report.html').read_text()
report+='<h2>Self-guidance: measured target response after four steps</h2><p>One guidance update followed by three ordinary updates. Target is relative to the ordinary trajectory at the matching next state. Inspect whether the intended direction persists; the local gradient check alone does not guarantee this.</p>'+pd.DataFrame(D_SG_OUTCOMES).to_html(index=False)
(D_OUT/'report.html').write_text(report)



In [ ]:
# 15. Conclusions from this dry run and artifact completeness.
assert len(D_PROBES)==3 and all(len(r['directions'])==13 for r in D_PROBES.values())
assert len(D_ENDPOINTS)==4 and len(D_JEV_RESULTS)==3 and all('answers' in r for r in D_JEV_RESULTS)
assert all((D_OUT/f'computations_{i}.pt').exists() for i in [50,65,80])
summary='''### What this run established
- All four mechanisms execute and produce nonzero, reproducible changes. Neutral attention matches the installed processor exactly. Ordinary repeat probes match exactly.
- Head directions differ: at checkpoint 50, head 5 has cosine -0.31 with all-head PAG. Joint heads 0+1 differ from the sum of their individual noise-prediction directions (RMS residual about 0.000615). This is an interaction measurement, not an anatomical attribution.
- Self-guidance has a real controllable internal target. At checkpoint 50, after four steps the attention-area proxy is about 0.0686 for ordinary continuation, 0.0767 after +10% guidance, and 0.0605 after -10% guidance. The intended direction also persists at the other two checkpoints. This is attention extent, not measured hand size or finger correctness. The single-branch property readout has a small numerical offset from the original CFG-batched reference.
- FreeU produces the largest final RGB change in these chosen settings. Its feature trace shows backbone amplification and low-frequency skip attenuation at the first two upsampling stages.
- Endpoint visual inspection shows mostly subtle changes relative to ordinary continuation, without an obvious anatomical repair. Original-source versus ordinary-continuation changes come from re-noising and denoising and must not be credited to an intervention.
- Jev chose the same four follow-up probes at all three checkpoints: PAG strength sweep; opposing heads; self-guidance target response; FreeU backbone/skip factorial. These choices were logged after the fixed mechanism runs. No adaptive Jev image-control gain has been demonstrated.

**Scope:** SD1.5 SoftPAG head diagnostic, not the complete HeadHunter search. Single-layer self-guidance with a declared size target and normalized gain, not a full reproduction of Imagen self-guidance. All images and intermediate computations are retained; no selected winner.
'''
display(Markdown(summary))
(D_OUT/'findings.md').write_text(summary)
print('Completed:',39,'matched intervention probes;',4,'full continuations;',3,'Jev requests. Outputs:',D_OUT)
display(HTML(f'<a target="_blank" href="{D_REPORT_URL}">Open the complete image and computation report</a>'))
